# Calculate the Average daily visit rate for three groups
Author: Callie Clark

In [ ]:
!pip install pydeck -q -q
exec(open('../EV00_settings.py').read())
from shapely import wkt
from scipy.spatial import cKDTree 

In [ ]:
import pandas as pd
from datetime import datetime as dt, timedelta
for current_city in ['SF','Boston','Seattle','Denver']:#,'Seattle','Denver','Boston'
    start_dt = dt.strptime(start_date, '%Y%m%d')
    end_dt = dt.strptime(end_date, '%Y%m%d')

    current_date = start_dt

    poi_matrix = None
    flag_list = []

    while current_date <= end_dt:

        date_str = current_date.strftime('%Y%m%d')
        print(date_str)

        df_iter = pd.read_csv(
            f'{current_city}_poi_stop_counts_{date_str}.csv',
           # usecols=['cuebiq_id','placekey','ev_driver','charge_day']
        )
        df_iter=df_iter[df_iter['category']!='no poi'].copy()
        df_iter=df_iter[df_iter['category']!='Other'].copy()

        # -------------------------
        # daily POI counts
        # -------------------------
        daily_counts = df_iter.groupby('cuebiq_id')['placekey'].count()
        daily_counts.name = date_str
        
        daily_counts_df=pd.DataFrame(daily_counts)
        if poi_matrix is None:
            poi_matrix = daily_counts_df.copy()
        else:
            poi_matrix = poi_matrix.join(daily_counts_df, how='outer')

        # -------------------------
        # collect flags
        # -------------------------
        daily_counts_df.columns=['poi_stops']
        df_iter=df_iter.merge(daily_counts_df,left_on='cuebiq_id',right_index=True)

        df_iter['date']= date_str
        flags = df_iter[['cuebiq_id','ev_driver','charge_day','tot_stops',
       'active_day','poi_stops','date']].drop_duplicates()
        flag_list.append(flags)

        current_date += timedelta(days=1)
        

        

    poi_matrix = poi_matrix.fillna(0).astype('int16')

    # combine flags
    user_flags = pd.concat(flag_list).drop_duplicates(['cuebiq_id','date']).set_index('cuebiq_id')

    poi_matrix.to_csv(f'{current_city}_poi_matrix_stops_filtered.csv')
    user_flags.to_csv(f'{current_city}_user_poi_stops_filtered.csv')


In [ ]:


def compute_poi_stats(user_df, debug=False):

    # keep only active days
    user_df['active_day']=(user_df.tot_stops>2)*1
    df = user_df[(user_df['active_day'] == 1)&(user_df['tot_pings'] >= 36)&(user_df['tot_pings'] < 2500)].copy()

    if debug:
        print("Total active user-days:", len(df))

    # masks
    masks = {
        "non_ev": df['ev_driver'] == 0,
        "ev_charge": (df['ev_driver'] == 1) & (df['charge_day'] == 1),
        "ev_noncharge": (df['ev_driver'] == 1) & (df['charge_day'] == 0)
    }

    dist = {}
    stats = {}

    for group, mask in masks.items():

        vals = df.loc[mask, 'poi_stops'].dropna().values
        dist[group] = vals

        n = len(vals)

        if n == 0:
            stats[group] = {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "n": 0}
            continue

        mean = vals.mean()
        std = vals.std()

        ci = 1.96 * std / np.sqrt(n)

        stats[group] = {
            "mean": mean,
            "ci_low": mean - ci,
            "ci_high": mean + ci,
            "n": n
        }

        if debug:
            print(group, "n =", n, "mean =", mean)

    return stats, dist

def compute_avg_poi_per_active_day(user_df):
    user_df['active_day']=(user_df.tot_stops>2)*1
    df = user_df[(user_df['active_day'] == 1)&(user_df['tot_pings'] >= 36)].copy()



    df = df.dropna(subset=['poi_stops'])

    # masks
    non_ev = df['ev_driver'] == 0
    ev_charge = (df['ev_driver'] == 1) & (df['charge_day'] == 1)
    ev_noncharge = (df['ev_driver'] == 1) & (df['charge_day'] == 0)

    # totals
    totals = {
        "non_ev": df.loc[non_ev, 'poi_stops'].sum(),
        "ev_charge": df.loc[ev_charge, 'poi_stops'].sum(),
        "ev_noncharge": df.loc[ev_noncharge, 'poi_stops'].sum()
    }

    # counts
    counts = {
        "non_ev": non_ev.sum(),
        "ev_charge": ev_charge.sum(),
        "ev_noncharge": ev_noncharge.sum()
    }

    # averages
    averages = {
        k: totals[k] / counts[k] if counts[k] > 0 else np.nan
        for k in totals
    }

    return averages, totals, counts

import matplotlib.pyplot as plt
import numpy as np

def plot_city_poi_means(city_stats, title="Average POI Visits per Day", fontsize=14):

    groups = ["non_ev", "ev_charge", "ev_noncharge"]
    labels = ["Non-EV Drivers", "EV Drivers \n (Charging Day)", "EV Drivers \n (Non-Charging Day)"]

    # Colors you can edit later
    color_dict = {
            "SF": "#efac60",
            "Seattle": "#598fa0",
            "Boston": "#455379",
            "Denver": "#b24955"
        }

    cities = list(city_stats.keys())
    y_pos = np.arange(len(groups))

    # horizontal offsets so cities don't overlap
    offsets = np.linspace(-0.2, 0.2, len(cities))

    plt.figure(figsize=(8,5))

    for i, city in enumerate(cities):

        stats = city_stats[city]

        means = [stats[g]["mean"] for g in groups]
        ci_low = [stats[g]["ci_low"] for g in groups]
        ci_high = [stats[g]["ci_high"] for g in groups]

        lower_err = np.array(means) - np.array(ci_low)
        upper_err = np.array(ci_high) - np.array(means)
        errors = np.vstack([lower_err, upper_err])

        plt.errorbar(
            means,
            y_pos + offsets[i],
            xerr=errors,
            fmt='o',
            capsize=4,
            label=city,
            color=color_dict.get(city, "black")
        )

    plt.yticks(y_pos, labels, fontsize=fontsize)
    plt.xlabel("POI Visits per Active Day", fontsize=fontsize)


    plt.xticks(fontsize=fontsize)
    plt.legend(fontsize=fontsize-1)

    plt.tight_layout()
    plt.show()

In [ ]:
city_stats = {}
for current_city in ['Seattle','Denver','SF','Boston']:#'Boston','SF','Seattle','Denver']:
    print(current_city)
    
    user_df_=pd.read_csv(f'{current_city}_user_poi_stops_filtered.csv')
    user_df_=user_df_.reset_index()

    visit_count_3 = pd.read_csv(data_path+f'raw/user_daily_ping_df_{current_city}_3.csv',index_col=0)
    visit_count_4 = pd.read_csv(data_path+f'raw/user_daily_ping_df_{current_city}_4.csv',index_col=0)
    visit_count_5 = pd.read_csv(data_path+f'raw/user_daily_ping_df_{current_city}_5.csv',index_col=0)
    visit_count= pd.concat([visit_count_3,visit_count_4,visit_count_5],axis=1)
    visit_count.columns = visit_count.columns.astype(str).str.replace('-', '')

    visit_df=visit_count.melt(ignore_index=False).dropna()
    visit_df.rename(columns={'variable':'date','value':'tot_pings'},inplace=True)
    visit_df=visit_df.reset_index()
    visit_df.head()

    visit_df.date=visit_df.date.astype('str')
    user_df_.date=user_df_.date.astype('str')
    visit_df.cuebiq_id=visit_df.cuebiq_id.astype('str')
    user_df_.cuebiq_id=user_df_.cuebiq_id.astype('str')

    user_df=user_df_.merge(visit_df,on=['cuebiq_id','date'])
    del visit_df
    del user_df_
    
    user_df.to_csv(f'{current_city}_daily_user_info.csv')


    poi_matrix=pd.read_csv(f'{current_city}_poi_matrix.csv',index_col=0)
    poi_matrix.columns = poi_matrix.columns.astype(str).str.replace('-', '')
    user_flag=pd.read_csv(f'{current_city}_user_flags.csv',index_col=0)
    user_flag['date'] = user_flag['date'].astype(str)
    
    stats, distributions = compute_poi_stats(user_df)
    del user_df
    print(stats)
    city_stats[current_city]=stats


plot_city_poi_means(
    city_stats,
    title="Average POI Visits per Active",
    fontsize=14
)
